In [56]:
import os
from dotenv import load_dotenv
from IPython.display import Markdown
import json
from statistics import mean
import re
import ast

from anthropic import Anthropic

In [40]:
load_dotenv(override=True)

anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")

client = Anthropic(api_key=anthropic_api_key)

model = "claude-sonnet-4-5-20250929"

In [20]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }

    if system:
        params["system"] = system

    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    message = client.messages.create(**params)
    return message.content[0].text

In [74]:
def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
        "format": "json" or "python" or "regex",
        "solution_criteria": "Key criteria for evaluating the solution"
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])

    return json.loads(text)

In [62]:
dataset = generate_dataset()
print(dataset)

[{'task': 'Write a regex pattern that validates AWS S3 bucket names according to AWS naming rules: 3-63 characters long, only lowercase letters, numbers, hyphens, and periods, must start and end with a letter or number, and cannot have consecutive periods or periods adjacent to hyphens.', 'format': 'regex'}, {'task': "Create a JSON object for an AWS IAM policy that allows read-only access to a specific S3 bucket named 'my-company-logs' and all objects within it.", 'format': 'json'}, {'task': 'Write a Python function that takes an AWS ARN string as input and returns a dictionary with the parsed components: partition, service, region, account_id, and resource.', 'format': 'python'}]


In [23]:
for item in dataset:
    print(item["task"])
    print()

Write a regular expression to validate AWS S3 bucket names. S3 bucket names must be between 3 and 63 characters, contain only lowercase letters, numbers, hyphens, and periods, start and end with a letter or number, and cannot contain consecutive periods or end with a hyphen.

Write a Python function that parses an AWS CloudFormation template (JSON string) and returns a list of all resource logical IDs and their types.

Write a JSON object that represents an AWS IAM policy allowing a user to read and list objects in a specific S3 bucket named 'my-app-bucket' but deny deletion permissions.



In [75]:
for i, item in enumerate(dataset, 1):
    print(f"{i}. {item['task']}\n")

1. Write a regex pattern that validates an AWS S3 bucket name according to AWS rules: 3-63 characters long, only lowercase letters, numbers, hyphens, and periods, must start and end with a letter or number, and cannot have consecutive periods.

2. Create a JSON object representing an AWS IAM policy that allows read-only access to a specific S3 bucket named 'my-data-bucket', including permissions to list objects and get objects.

3. Write a Python function that takes an AWS ARN (Amazon Resource Name) string as input and returns a dictionary containing the parsed components: partition, service, region, account-id, and resource.



In [25]:
print(json.dumps(dataset, indent=2))

[
  {
    "task": "Write a regular expression to validate AWS S3 bucket names. S3 bucket names must be between 3 and 63 characters, contain only lowercase letters, numbers, hyphens, and periods, start and end with a letter or number, and cannot contain consecutive periods or end with a hyphen."
  },
  {
    "task": "Write a Python function that parses an AWS CloudFormation template (JSON string) and returns a list of all resource logical IDs and their types."
  },
  {
    "task": "Write a JSON object that represents an AWS IAM policy allowing a user to read and list objects in a specific S3 bucket named 'my-app-bucket' but deny deletion permissions."
  }
]


In [76]:
dataset = generate_dataset()

with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [64]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the followint task:

{test_case["task"]}

* Respond only with Python, JSON, or a plain Regex
* Do not add any comments, commentary, or explanation
"""
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code")
    output = chat(messages, stop_sequences=["```"])
    return output

In [77]:
def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert AWS code reviewer.  Your task is to evaluate the dollowing AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Criteria you should use to evaluate the solution:
<criteria>
{test_case["solution_criteria"]}
</criteria>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses: An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10   

Respond with JSON.  Keep your responses concise and direct.
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string[],
    ""score": number
}}
    """
    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

In [78]:
def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0

def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0

def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0

def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)

In [79]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    # Grading
    model_grade = model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    syntax_score = grade_syntax(output, test_case)

    score = (model_score + syntax_score) / 2

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning
    }

In [81]:

def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(average_score)

    return results

In [82]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

final_results = run_eval(dataset)

8.166666666666666


In [83]:
print(json.dumps(final_results, indent=2))

[
  {
    "output": "\ndef extract_service_from_arn(arn: str) -> str:\n    parts = arn.split(':')\n    if len(parts) >= 3 and parts[0] == 'arn':\n        return parts[2]\n    return ''\n",
    "test_case": {
      "task": "Write a Python function that takes an AWS ARN string and extracts the service name from it. For example, given 'arn:aws:s3:::my-bucket' it should return 's3'.",
      "format": "python",
      "solution_criteria": "Function correctly parses ARN format, extracts the service component (3rd colon-separated field), handles edge cases like invalid ARNs, and returns string or appropriate error"
    },
    "score": 8.0,
    "reasoning": "The solution handles the basic happy path correctly and performs minimal validation. However, it lacks robust error handling - returning empty strings for invalid input makes it difficult for calling code to distinguish between errors and edge cases. The function should either raise descriptive exceptions for invalid ARNs or document the em